In [25]:
# Finance set

tableName='finance_markers'
filterColumn='investigationState'
filterValue='Complete'
classifierColumn='fraudtype'

In [26]:
# Some data exploration

%load_ext sql
%sql trino://streambased-server:8080/kafka
%sql SELECT * FROM kafka.streambased.{{tableName}} LIMIT 10


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Running query in 'trino://streambased-server:8080/kafka'

fraudtype,transcountabove1stddev,multipleinternationtransactions,multipleaddresschanges,transamountabove1stddev,complaints,above10adjustments,multipledocumentverificationattempts,accountmorethan6monthsold,investigationstate,_partition_id,_partition_offset,_message_corrupt,_message,_headers,_message_length,_key_corrupt,_key,_key_length,_timestamp
Credit Card Fraud,1,0,0,1,1,1,1,1,Complete,0,0,False,"    ""Credit Card Fraud  Complete",{},40,False,None,0,2024-02-15 13:08:46.322000
Payment Fraud,1,1,1,0,0,1,1,0,Complete,0,1,False,    Payment Fraud   Complete,{},36,False,None,0,2024-02-15 13:08:46.353000
Credit Card Fraud,1,0,0,1,1,1,1,0,Complete,0,2,False,"    ""Credit Card Fraud   Complete",{},40,False,None,0,2024-02-15 13:08:46.353000
Insurance Fraud,0,1,0,1,1,0,1,0,Complete,0,3,False,     Insurance Fraud    Complete,{},38,False,None,0,2024-02-15 13:08:46.354000
Credit Card Fraud,0,1,1,1,0,0,1,0,Complete,0,4,False,"    ""Credit Card Fraud    Complete",{},40,False,None,0,2024-02-15 13:08:46.355000
Tax Evasion,1,0,0,1,1,1,0,0,Complete,0,5,False,    Tax Evasion    Complete,{},34,False,None,0,2024-02-15 13:08:46.355000
Credit Card Fraud,0,1,1,1,1,1,1,0,Complete,0,6,False,"    ""Credit Card Fraud  Complete",{},40,False,None,0,2024-02-15 13:08:46.356000
Insurance Fraud,1,1,0,1,0,0,1,0,Complete,0,7,False,     Insurance Fraud    Complete,{},38,False,None,0,2024-02-15 13:08:46.356000
Tax Evasion,0,0,1,1,1,0,1,1,Complete,0,8,False,    Tax Evasion   Complete,{},34,False,None,0,2024-02-15 13:08:46.357000
Payment Fraud,0,0,1,0,0,0,0,1,Complete,0,9,False,    Payment Fraud      Complete,{},36,False,None,0,2024-02-15 13:08:46.358000


In [27]:
# Fetch our relevant dataset

result = %sql SELECT * FROM kafka.streambased.{{tableName}} WHERE {{filterColumn}}='{{filterValue}}'
import pandas as pd
df = result.DataFrame()

Running query in 'trino://streambased-server:8080/kafka'

In [28]:
# Build the model

import time
import pandas as pd
import pandas as pd
from joblib import dump, load
from river import tree, compose, metrics

# Assuming df is your DataFrame after loading and preprocessing
features = [column for column in df.columns if column not in [filterColumn, '_partition_id', '_partition_offset',
                                                               '_message_corrupt', '_message', '_headers',
                                                               '_message_length', '_key_corrupt', '_key',
                                                               '_key_length', '_timestamp', 'cuisine']]

# Initialize or load the model
model_path = "hoeffding_tree_model.joblib"
try:
    model = load(model_path)
    print("Model loaded successfully.")
except FileNotFoundError:
    model = tree.HoeffdingTreeClassifier()
    print("New model initialized.")

# Prepare a metric to track performance (optional)
metric = metrics.Accuracy()

total_trained_rows = 0

# Simulate a data stream and train the model incrementally
for index, row in df.iterrows():
    # Extract features and target variable for the current row
    x = row[features].to_dict()
    y = row[classifierColumn]
    # Update the model and optionally update metric
    model.learn_one(x, y)
    # metric = metric.update(y, model.predict_one(x))  # Uncomment if you want to track performance
    total_trained_rows += 1

# Save the updated model
dump(model, model_path)
print(f"Model updated and saved. Total trained rows: {total_trained_rows}.")

# Example of making a prediction with the updated model
# x_new = {feature: value, ...}  # New data point's features
# y_pred = model.predict_one(x_new)


Model loaded successfully.
Model updated and saved. Total trained rows: 5002.


In [29]:
# Show model parameters

value_counts = df[classifierColumn].value_counts()
value_counts

fraudtype
Credit Card Fraud    1251
Identity Theft       1100
Insurance Fraud      1050
Payment Fraud         926
Tax Evasion           675
Name: count, dtype: int64